In [344]:
from pathlib import Path

from sklearn.model_selection import train_test_split

# Obtener todos los archivos .C en el directorio 
c_files = list(Path("soco-dataset/c").glob("*.c"))
java_files = list(Path("soco-dataset/java").glob("*.java"))

files = c_files + java_files

In [345]:
from transformers import AutoTokenizer, AutoModel
import torch

# Utilizar tokenizer y modelo de embedding UniXcoder
tokenizer = AutoTokenizer.from_pretrained("microsoft/unixcoder-base")
embedding_model = AutoModel.from_pretrained("microsoft/unixcoder-base")

print("Successfully imported tokenizer and model UNIXCODER")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Successfully imported tokenizer and model UNIXCODER


In [346]:
def tokenize_and_embed(filepath):
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        code = f.read()

    tokens = tokenizer(
        code,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = embedding_model(**tokens)

    embedding = outputs.last_hidden_state.mean(dim=1)

    return embedding

In [347]:
from tqdm.notebook import tqdm
import numpy as np

def save_embeddings(files):
    embeddings = []
    
    for file in tqdm(
        files,
        desc="🎀 KATSEYE EMBEDDINGS 🎀",
        unit="file"
    ):
        embedded_code = tokenize_and_embed(file)
        embeddings.append(embedded_code)

    embeddings = np.array(embeddings)
    embeddings = embeddings.squeeze(1)
    
    return embeddings

In [348]:
embeddings = save_embeddings(files)

🎀 KATSEYE EMBEDDINGS 🎀:   0%|          | 0/338 [00:00<?, ?file/s]

In [349]:
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(embeddings)

In [350]:
""" import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))

plt.imshow(sim_matrix, cmap="RdPu")
plt.colorbar(label="Cosine Similarity")

plt.title("Similarity Matrix")
plt.xlabel("File Index")
plt.ylabel("File Index")

plt.show() """

' import matplotlib.pyplot as plt\n\nplt.figure(figsize=(10, 8))\n\nplt.imshow(sim_matrix, cmap="RdPu")\nplt.colorbar(label="Cosine Similarity")\n\nplt.title("Similarity Matrix")\nplt.xlabel("File Index")\nplt.ylabel("File Index")\n\nplt.show() '

In [351]:
""" threshold = 0.95

rows, cols = np.where(sim_matrix > threshold)

plt.figure(figsize=(10, 10))
plt.scatter(cols, rows, s=10)
plt.xlabel("File")
plt.ylabel("File")
plt.title("Similar Pairs (>0.95)")
plt.show() """

' threshold = 0.95\n\nrows, cols = np.where(sim_matrix > threshold)\n\nplt.figure(figsize=(10, 10))\nplt.scatter(cols, rows, s=10)\nplt.xlabel("File")\nplt.ylabel("File")\nplt.title("Similar Pairs (>0.95)")\nplt.show() '

In [352]:
def sort_similarities(files, embeddings, sim_matrix):
    pairs = []
    
    for i in range(len(embeddings)):
        for j in range(i + 1, len(embeddings)):
            plagiarism = 0
            if sim_matrix[i][j] > 0.8:
                plagiarism = 1
            pairs.append((files[i], files[j], round(float(sim_matrix[i][j]), 4), plagiarism))
    
    pairs.sort(key=lambda x: x[2], reverse=True)

    return pairs

In [353]:
def create_dataframe(pairs, num_of_pairs):
    data_frame = pd.DataFrame(
    
        pairs[:num_of_pairs],
    
        columns=["file1", "file2", "similarity", "plagiarism_suspected"]
    
    )
    
    data_frame["file1"] = data_frame["file1"].apply(lambda x: x.name)
    data_frame["file2"] = data_frame["file2"].apply(lambda x: x.name)

    return data_frame
    

In [354]:
pairs = sort_similarities(files, embeddings, sim_matrix)

dataframe = create_dataframe(pairs, 60000)

display(dataframe)

,file1,file2,similarity,plagiarism_suspected
0,035.c,036.c,1.0000,1
1,005.java,006.java,0.9999,1
2,015.java,023.java,0.9990,1
3,043.java,251.java,0.9932,1
4,024.java,016.java,0.9927,1
...,...,...,...,...
56948,073.java,186.java,0.0825,0
56949,032.c,033.java,0.0822,0
56950,073.java,039.java,0.0818,0
56951,073.java,050.java,0.0798,0


In [355]:
def extract_features(code):
    return {
        "num_loops": count_loops(code),
        "num_ifs": count_ifs(code),
        "num_functions": count_functions(code),
        "num_variables": count_variables(code),
        "avg_function_length": avg_function_length(code)
    }

# Dataset Construction: Similarity Features

Usamos los archivos `.qrel` como ground truth real (en vez del threshold de coseno) y calculamos las métricas de similitud textual para cada par.

In [ ]:

import subprocess
subprocess.run(["pip", "install", "rapidfuzz", "-q"], capture_output=True)

import re
import math
import pandas as pd
import numpy as np
from collections import Counter
from pathlib import Path
from rapidfuzz.distance import Levenshtein as RapidLev
from rapidfuzz.distance import Jaro, JaroWinkler

# Cargar ground truth desde archivos .qrel
def load_ground_truth():
    plagiarism_pairs = set()
    for qrel_path in ["soco-dataset/SOCO14-c.qrel", "soco-dataset/SOCO14-java.qrel"]:
        with open(qrel_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 2:
                    a, b = parts
                    plagiarism_pairs.add((a, b))
                    plagiarism_pairs.add((b, a))
    return plagiarism_pairs

ground_truth = load_ground_truth()
print(f"Pares de plagio en ground truth: {len(ground_truth) // 2}")


In [ ]:

def read_code(filename):
    for folder in ["soco-dataset/c", "soco-dataset/java"]:
        path = Path(folder) / filename
        if path.exists():
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                return f.read()
    return ""

def normalize_code(code):
    # Eliminar comentarios de línea y bloque
    code = re.sub(r"//.*?\n|/\*.*?\*/", " ", code, flags=re.DOTALL)
    return re.sub(r"\s+", " ", code).strip().lower()

def tokenize(code):
    return re.findall(r"\w+", code)

def get_ngrams(tokens, n):
    return list(zip(*[tokens[i:] for i in range(n)]))

def jaccard(seq_a, seq_b):
    set_a, set_b = set(seq_a), set(seq_b)
    if not set_a and not set_b:
        return 1.0
    inter = len(set_a & set_b)
    union = len(set_a | set_b)
    return inter / union if union > 0 else 0.0

def dice(seq_a, seq_b):
    set_a, set_b = set(seq_a), set(seq_b)
    if not set_a and not set_b:
        return 1.0
    inter = len(set_a & set_b)
    denom = len(set_a) + len(set_b)
    return 2 * inter / denom if denom > 0 else 0.0

def cosine_bow(tokens_a, tokens_b):
    ca, cb = Counter(tokens_a), Counter(tokens_b)
    vocab = set(ca) | set(cb)
    dot = sum(ca[w] * cb[w] for w in vocab)
    mag_a = math.sqrt(sum(v ** 2 for v in ca.values()))
    mag_b = math.sqrt(sum(v ** 2 for v in cb.values()))
    return dot / (mag_a * mag_b) if mag_a and mag_b else 0.0

def compute_all_features(file1, file2):
    code1 = normalize_code(read_code(file1))
    code2 = normalize_code(read_code(file2))
    tokens1 = tokenize(code1)
    tokens2 = tokenize(code2)

    # Limitar tokens para que Levenshtein/Jaro sean razonablemente rápidos
    t1 = " ".join(tokens1[:300])
    t2 = " ".join(tokens2[:300])

    lev_dist = RapidLev.distance(t1, t2)
    max_len = max(len(t1), len(t2))
    lev_ratio = 1.0 - (lev_dist / max_len) if max_len > 0 else 1.0

    return {
        "jaccard_tokens":    jaccard(tokens1, tokens2),
        "jaccard_bigrams":   jaccard(get_ngrams(tokens1, 2), get_ngrams(tokens2, 2)),
        "jaccard_trigrams":  jaccard(get_ngrams(tokens1, 3), get_ngrams(tokens2, 3)),
        "dice":              dice(tokens1, tokens2),
        "levenshtein_dist":  lev_dist,
        "levenshtein_ratio": lev_ratio,
        "jaro":              Jaro.similarity(t1, t2),
        "jaro_winkler":      JaroWinkler.similarity(t1, t2),
        "cosine_similarity": cosine_bow(tokens1, tokens2),
    }

print("Funciones de similitud definidas correctamente.")


In [ ]:

from tqdm.notebook import tqdm

# Etiquetar pares usando ground truth real (no threshold de coseno)
df_labeled = dataframe.copy()
df_labeled["plagiarism"] = df_labeled.apply(
    lambda row: 1 if (row["file1"], row["file2"]) in ground_truth else 0,
    axis=1
)

positives = df_labeled[df_labeled["plagiarism"] == 1]
negatives = df_labeled[df_labeled["plagiarism"] == 0]

# Dataset balanceado: todos los positivos + 3x negativos muestreados aleatoriamente
n_neg = min(len(positives) * 3, len(negatives))
df_balanced = pd.concat(
    [positives, negatives.sample(n_neg, random_state=42)]
).reset_index(drop=True)

print(f"Positivos (plagio):     {len(positives)}")
print(f"Negativos (no plagio):  {n_neg}")
print(f"Total pares a procesar: {len(df_balanced)}")

# Calcular todas las features de similitud
features_list = []
for _, row in tqdm(df_balanced.iterrows(), total=len(df_balanced), desc="Calculando features"):
    feats = compute_all_features(row["file1"], row["file2"])
    feats["embedding_cosine"] = row["similarity"]   # coseno desde embeddings UniXcoder
    feats["plagiarism"] = row["plagiarism"]
    features_list.append(feats)

features_df = pd.DataFrame(features_list)
display(features_df)
print(f"\nDistribución de clases:\n{features_df['plagiarism'].value_counts()}")


## Modelo de Detección de Plagio

Entrenamos un Random Forest con las 10 features de similitud y evaluamos con métricas estándar de clasificación.

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

FEATURE_COLS = [
    "jaccard_tokens", "jaccard_bigrams", "jaccard_trigrams",
    "dice", "levenshtein_dist", "levenshtein_ratio",
    "jaro", "jaro_winkler", "cosine_similarity", "embedding_cosine"
]

X = features_df[FEATURE_COLS].values
y = features_df["plagiarism"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)
model.fit(X_train, y_train)

y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# --- Cross-validation ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=cv, scoring="f1", n_jobs=-1)
print(f"F1 CV (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# --- Reporte ---
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=["No Plagio", "Plagio"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")

# --- Matriz de confusión ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=["No Plagio", "Plagio"]).plot(
    ax=axes[0], cmap="RdPu", colorbar=False
)
axes[0].set_title("Matriz de Confusión")

# --- Curva ROC ---
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color="#c77dff", lw=2,
             label=f"AUC = {roc_auc_score(y_test, y_proba):.4f}")
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("Curva ROC")
axes[1].legend()

# --- Feature importances ---
importances = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values()
importances.plot(kind="barh", ax=axes[2], color="#c77dff")
axes[2].set_title("Importancia de Features")
axes[2].set_xlabel("Importancia")

plt.tight_layout()
plt.show()
